In [1]:
!pip install datasets evaluate transformers[sentencepiece]

In [2]:
# load the data
from datasets import load_dataset

data = load_dataset('lewtun/github-issues', split='train')
data

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Repo card metadata block was not found. Setting CardData to empty.


Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 3019
})

In [3]:
# remove the rows where there are no comments and have a pull reuests

data = data.filter(
    lambda x:(x['is_pull_request'] == False and len(x['comments']) > 0)
)
data

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'pull_request', 'body', 'timeline_url', 'performed_via_github_app', 'is_pull_request'],
    num_rows: 808
})

In [4]:
# keep only the columns we want to keep
columns = data.column_names
cols_to_keep = ['title', 'body', 'html_url', 'comments']
cols_to_remove = list(set(columns) - set(cols_to_keep))
cols_to_remove

['node_id',
 'labels',
 'labels_url',
 'updated_at',
 'user',
 'timeline_url',
 'number',
 'repository_url',
 'closed_at',
 'created_at',
 'locked',
 'performed_via_github_app',
 'author_association',
 'state',
 'milestone',
 'url',
 'pull_request',
 'active_lock_reason',
 'id',
 'comments_url',
 'is_pull_request',
 'events_url',
 'assignee',
 'assignees']

In [5]:
data = data.remove_columns(cols_to_remove)
data

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 808
})

In [6]:
print(data[0]['comments'])
print("Type of comments field:", type(data[0]['comments']))

['Cool, I think we can do both :)', '@lhoestq now the 2 are implemented.\r\n\r\nPlease note that for the the second protection, finally I have chosen to protect the master branch only from **merge commits** (see update comment above), so no need to disable/re-enable the protection on each release (direct commits, different from merge commits, can be pushed to the remote master branch; and eventually reverted without messing up the repo history).']
Type of comments field: <class 'list'>


 Because our comments column is currently a list of comments for each issue, we need to “explode” the column so that each row consists of an (html_url, title, body, comment) tuple. In Pandas we can do this with the DataFrame.explode() function, which creates a new row for each element in a list-like column, while replicating all the other column values.

In [7]:
data.set_format('pandas')
df = data[:]

In [8]:
df.head()

,html_url,title,comments,body
0,https://github.com/huggingface/datasets/issues...,Protect master branch,"[Cool, I think we can do both :), @lhoestq now...",After accidental merge commit (91c55355b634d0d...
1,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,[Hi ! I guess the caching mechanism should hav...,## Describe the bug\r\nAfter upgrading to data...
2,https://github.com/huggingface/datasets/issues...,OSCAR unshuffled_original_ko: NonMatchingSplit...,[I tried `unshuffled_original_da` and it is al...,## Describe the bug\r\n\r\nCannot download OSC...
3,https://github.com/huggingface/datasets/issues...,load_dataset using default cache on Windows ca...,"[Hi @daqieq, thanks for reporting.\r\n\r\nUnfo...",## Describe the bug\r\nStandard process to dow...
4,https://github.com/huggingface/datasets/issues...,to_tf_dataset keeps a reference to the open da...,"[I did some investigation and, as it seems, th...",To reproduce:\r\n```python\r\nimport datasets ...


In [9]:
len(df['comments'][0])

2

In [10]:
comments_df = df.explode("comments", ignore_index=True)
comments_df.head()

,html_url,title,comments,body
0,https://github.com/huggingface/datasets/issues...,Protect master branch,"Cool, I think we can do both :)",After accidental merge commit (91c55355b634d0d...
1,https://github.com/huggingface/datasets/issues...,Protect master branch,@lhoestq now the 2 are implemented.\r\n\r\nPle...,After accidental merge commit (91c55355b634d0d...
2,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,Hi ! I guess the caching mechanism should have...,## Describe the bug\r\nAfter upgrading to data...
3,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,"If it's easy enough to implement, then yes ple...",## Describe the bug\r\nAfter upgrading to data...
4,https://github.com/huggingface/datasets/issues...,Backwards compatibility broken for cached data...,Well it can cause issue with anyone that updat...,## Describe the bug\r\nAfter upgrading to data...


In [11]:
#  Switch back to datasets
from datasets import Dataset

comments_ds = Dataset.from_pandas(comments_df)
comments_ds

Dataset({
    features: ['html_url', 'title', 'comments', 'body'],
    num_rows: 2964
})

In [12]:
"""
Preferred method to explode using map:

def explode_comments(batch):
    # 1. Open up the comments list (the "explode" part)
    # We create a single list of all individual comments in this batch
    comments = [c for sublist in batch["comments"] for c in sublist]

    # 2. Duplicate specific metadata (the "copy" part)
    # We calculate how many times to repeat each row's data
    repeats = [len(c_list) for c_list in batch["comments"]]

    # Just repeat the columns you actually care about
    # This keeps the code cleaner than a generic loop
    return {
        "comments": comments,
        "issue_id": [val for val, count in zip(batch["id"], repeats) for _ in range(count)],
        "title": [val for val, count in zip(batch["title"], repeats) for _ in range(count)],
    }

# Apply the map and drop ALL old columns to let the new structure take over
exploded_dataset = issues_dataset.map(
    explode_comments,
    batched=True,
    remove_columns=issues_dataset.column_names
)

"""

'\nPreferred method to explode using map:\n\ndef explode_comments(batch):\n    # 1. Open up the comments list (the "explode" part)\n    # We create a single list of all individual comments in this batch\n    comments = [c for sublist in batch["comments"] for c in sublist]\n    \n    # 2. Duplicate specific metadata (the "copy" part)\n    # We calculate how many times to repeat each row\'s data\n    repeats = [len(c_list) for c_list in batch["comments"]]\n    \n    # Just repeat the columns you actually care about\n    # This keeps the code cleaner than a generic loop\n    return {\n        "comments": comments,\n        "issue_id": [val for val, count in zip(batch["id"], repeats) for _ in range(count)],\n        "title": [val for val, count in zip(batch["title"], repeats) for _ in range(count)],\n    }\n\n# Apply the map and drop ALL old columns to let the new structure take over\nexploded_dataset = issues_dataset.map(\n    explode_comments, \n    batched=True, \n    remove_columns=iss

In [13]:
comments_ds = comments_ds.map(
    lambda x: {"comment_length": len(x['comments'].split())}
)
comments_ds

Map:   0%|          | 0/2964 [00:00<?, ? examples/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length'],
    num_rows: 2964
})

In [14]:
comments_ds = comments_ds.filter(
    lambda x: x['comment_length'] > 15
)
comments_ds

Filter:   0%|          | 0/2964 [00:00<?, ? examples/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length'],
    num_rows: 2175
})

In [15]:
def concat_text(example):
    return {
        "text": example['title']
        + " \n "
        + example['body']
        + "\n"
        + example['comments']
    }

comments_ds = comments_ds.map(concat_text)

Map:   0%|          | 0/2175 [00:00<?, ? examples/s]

In [16]:
comments_ds

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length', 'text'],
    num_rows: 2175
})

Load the model to create the sentence emb

In [17]:
from transformers import AutoTokenizer, AutoModel

model_ckpt = "sentence-transformers/multi-qa-mpnet-base-dot-v1"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
import torch

device = torch.device("cuda")
model.to(device)

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [19]:
# we simply collect the last hidden state for the special [CLS] token.
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

In [20]:
def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list, padding=True, truncation=True, return_tensors="pt"
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)
    return cls_pooling(model_output)

In [21]:
embedding = get_embeddings(comments_ds['text'][0])
embedding.shape

torch.Size([1, 768])

In [22]:
# create a new embedding column
embeddings_dataset = comments_ds.map(
    lambda x: {"embeddings": get_embeddings(x["text"]).detach().cpu().numpy()[0]}
)

Map:   0%|          | 0/2175 [00:00<?, ? examples/s]

In [23]:
embeddings_dataset

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length', 'text', 'embeddings'],
    num_rows: 2175
})

In [24]:
!pip install faiss-cpu

In [25]:
embeddings_dataset.add_faiss_index(column='embeddings')

  0%|          | 0/3 [00:00<?, ?it/s]

Dataset({
    features: ['html_url', 'title', 'comments', 'body', 'comment_length', 'text', 'embeddings'],
    num_rows: 2175
})

In [27]:
# lets try it out

question = "How can I load a dataset offline?"
ex_embd = get_embeddings([question]).cpu().detach().numpy()
ex_embd.shape

(1, 768)

In [34]:
# get the top k=3 results
scores, samples = embeddings_dataset.get_nearest_examples(
    'embeddings', ex_embd, k=3
)

In [35]:
# convert to dataframe
import pandas as pd

samples_df = pd.DataFrame.from_dict(samples)
samples_df["scores"] = scores
samples_df.sort_values("scores", ascending=False, inplace=True)

In [36]:
samples_df

,html_url,title,comments,body,comment_length,text,embeddings,scores
2,https://github.com/huggingface/datasets/issues...,Discussion using datasets in offline mode,I opened a PR that allows to reload modules th...,"`datasets.load_dataset(""csv"", ...)` breaks if ...",179,Discussion using datasets in offline mode \n `...,"[-0.4716479778289795, 0.2902272641658783, -0.0...",24.148989
1,https://github.com/huggingface/datasets/issues...,Discussion using datasets in offline mode,"> here is my way to load a dataset offline, bu...","`datasets.load_dataset(""csv"", ...)` breaks if ...",76,Discussion using datasets in offline mode \n `...,"[-0.4992601275444031, 0.22699788212776184, -0....",22.894003
0,https://github.com/huggingface/datasets/issues...,Discussion using datasets in offline mode,"here is my way to load a dataset offline, but ...","`datasets.load_dataset(""csv"", ...)` breaks if ...",47,Discussion using datasets in offline mode \n `...,"[-0.4902574121952057, 0.22889623045921326, -0....",22.406656


In [37]:
for _, row in samples_df.iterrows():
    print(f"COMMENT: {row.comments}")
    print(f"SCORE: {row.scores}")
    print(f"TITLE: {row.title}")
    print(f"URL: {row.html_url}")
    print("=" * 50)
    print()

COMMENT: I opened a PR that allows to reload modules that have already been loaded once even if there's no internet.

Let me know if you know other ways that can make the offline mode experience better. I'd be happy to add them :) 

I already note the "freeze" modules option, to prevent local modules updates. It would be a cool feature.

----------

> @mandubian's second bullet point suggests that there's a workaround allowing you to use your offline (custom?) dataset with `datasets`. Could you please elaborate on how that should look like?

Indeed `load_dataset` allows to load remote dataset script (squad, glue, etc.) but also you own local ones.
For example if you have a dataset script at `./my_dataset/my_dataset.py` then you can do
```python
load_dataset("./my_dataset")
```
and the dataset script will generate your dataset once and for all.

----------

About I'm looking into having `csv`, `json`, `text`, `pandas` dataset builders already included in the `datasets` package, so that 

The 2nd one seems the most similar to our interrogation.

## Final Notes

In this notebook, we built a semantic search system using the Hugging Face ecosystem and FAISS. Instead of relying on traditional keyword matching, this approach uses transformer-based embeddings to understand the semantic meaning of text and retrieve the most relevant results.

The workflow included loading and cleaning a dataset of GitHub issues, preparing the text data, generating embeddings using a pretrained sentence-transformer model, and indexing those embeddings using FAISS for efficient similarity search.

This project demonstrates how modern NLP tools can be used to build powerful search systems capable of understanding context and meaning within large collections of text.

Such systems are widely used in real-world applications like question answering systems, developer documentation search, knowledge retrieval systems, and AI assistants.
